In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from transformers import CLIPTokenizer, CLIPTextModel, CLIPVisionModel

device = "cuda"

In [5]:
MODEL_NAME = "openai/clip-vit-base-patch32"
CACHE_DIR = "./hf_models"

tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
vision_encoder = CLIPVisionModel.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR).to(device)
text_encoder = CLIPTextModel.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.final_layer_norm.bias                             | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.embeddings.class_embedding                        | UNEXPECTED |  | 
vision_model.encoder.l

In [ ]:
vision_hidden_dim = vision_encoder.config.hidden_size
text_hidden_dim = text_encoder.config.hidden_size

print("Vision hidden dimension:", vision_hidden_dim)
print("Text hidden dimension:", text_hidden_dim)
print("Tokenizer vocabulary size:", tokenizer.vocab_size)
print("Maximum text length:", tokenizer.model_max_length)

Vision hidden dimension: 768
Text hidden dimension: 512
Tokenizer vocabulary size: 49408
Maximum text length: 77


In [ ]:
class InhuClib(nn.Module):
    def __init__(self, v_encoder, t_encoder, tokenizer)